# Downloading the GLORYS12 Input for the 4-D Examples

This notebook fetches the input for the other three: daily potential temperature from GLORYS12V1, the 1/12° Copernicus
Marine reanalysis (`GLOBAL_MULTIYEAR_PHY_001_030`), over the northeast Pacific (30°N-60°N, 170°W-120°W) from the
surface to the layer containing 1000 m. That is 36 levels and 7.81 M voxels per day, or 340 GB as `float32` over the
30 years 1993-2022.

It writes one zarr store per year plus two static stores. `glorys12_static.zarr` contains the geometry, and
`glorys12_valid.zarr` contains the cells that are ocean and finite on every day. A few cells that the static mask calls
ocean are NaN in `thetao`, so the examples use `valid` as the ocean domain.

The download is resumable. Each year is written in 30-day blocks, a finished block leaves a marker, and a finished
year leaves `thetao_<year>.complete`. Rerunning any cell skips what is on disk.

Compute: one job with 8 CPUs, 12 GB and internet access, fetching one year at a time. A year takes about 16 minutes.
This notebook runs in its own environment with `copernicusmarine` installed (`pip install copernicusmarine ipykernel`)
and needs a prior `copernicusmarine login`. It does not use marEx.

## Setup

In [ ]:
import os
import time
from getpass import getuser
from pathlib import Path

import copernicusmarine as cm
import numpy as np
import pandas as pd
import xarray as xr
from numcodecs import Blosc

In [ ]:
scratch_dir = Path("/scratch") / getuser()[0] / getuser()
input_dir = scratch_dir / "marEx_data" / "input" / "glorys12_nepacific"

years = range(1993, 2023)  # the baseline of 01_preprocess_extremes.ipynb
lat_range, lon_range = (30.0, 60.0), (-170.0, -120.0)  # the product stores longitude in -180..180
depth_target = 1000.0  # keep every level down to the layer containing this depth (m)

time_chunk = 10  # days per zarr chunk
block_days = 30  # days per resumable download block, a multiple of time_chunk

dataset_daily = "cmems_mod_glo_phy_my_0.083deg_P1D-m"
dataset_static = "cmems_mod_glo_phy_my_0.083deg_static"

In [ ]:
input_dir.mkdir(parents=True, exist_ok=True)
compressor = Blosc(cname="zstd", clevel=3, shuffle=Blosc.BITSHUFFLE)
box = dict(
    minimum_latitude=lat_range[0], maximum_latitude=lat_range[1], minimum_longitude=lon_range[0], maximum_longitude=lon_range[1]
)

## 1. Static Geometry

The product gives the layer thickness `e3t` as one standard profile (no partial cells) and no interface depths, so the
interfaces are the cumulative thickness from 0 m. Levels 34 and 35 span 829.5-978.7 m and 978.7-1150.0 m, so the 1000 m
target keeps levels 0 to 35. The cell area is `e1t × e2t`, within 0.003 % of $R^2 \cos\phi\,\Delta\phi\,\Delta\lambda$
at $R$ = 6378 km.

In [ ]:
static_store = input_dir / "glorys12_static.zarr"
static_done = input_dir / "glorys12_static.zarr.complete"


def open_static(part):
    ds = cm.open_dataset(dataset_id=dataset_static, dataset_part=part, **box)
    return ds.rename({"latitude": "lat", "longitude": "lon"}).load()


if not static_done.exists():
    bathy, metrics = open_static("bathy"), open_static("coords")
    e3t = metrics["e3t"].values
    interfaces = np.concatenate([[0.0], np.cumsum(e3t.astype("f8"))])
    n_levels = int(np.searchsorted(interfaces, depth_target, side="right"))
    static = xr.Dataset(
        {
            "mask": bathy["mask"].isel(depth=slice(0, n_levels)).astype("int8"),
            "depth_bnds": (("depth", "bnds"), np.stack([interfaces[:n_levels], interfaces[1 : n_levels + 1]], axis=1)),
            "e3t": ("depth", e3t[:n_levels].astype("f8")),
            "cell_area": (metrics["e1t"] * metrics["e2t"]).astype("f8"),
            "deptho": bathy["deptho"],
        },
        coords={"depth": bathy["depth"].values[:n_levels].astype("f8"), "lat": bathy["lat"].values, "lon": bathy["lon"].values},
    )
    static["depth"].attrs = {"units": "m", "positive": "down", "long_name": "level centre depth"}
    static["depth_bnds"].attrs = {"units": "m", "long_name": "layer interfaces, positive down"}
    static["cell_area"].attrs = {"units": "m2", "long_name": "e1t * e2t"}
    static.to_zarr(static_store, mode="w", consolidated=True)
    static_done.touch()

static = xr.open_zarr(static_store)
n_levels = static.sizes["depth"]
print(
    f"{n_levels} levels, bottom interface {static.depth_bnds.values[-1, 1]:.1f} m, {static.sizes['lat']} x {static.sizes['lon']} cells"
)

## 2. Daily Temperature

Each year is first created as an empty store, then filled one block at a time with region writes. The source packs
`thetao` as integers, so the stored `float32` drops its `scale_factor` and `add_offset`. The stores are chunked 10 days
deep with depth, latitude and longitude whole.

In [ ]:
def fetch_year(year):
    store = input_dir / f"thetao_{year}.zarr"
    done = input_dir / f"thetao_{year}.complete"
    if done.exists():
        return "complete"

    ds = cm.open_dataset(
        dataset_id=dataset_daily,
        variables=["thetao"],
        start_datetime=f"{year}-01-01",
        end_datetime=f"{year}-12-31",
        **box,
    )
    ds = ds.rename({"latitude": "lat", "longitude": "lon"}).isel(depth=slice(0, n_levels))
    ds["thetao"] = ds["thetao"].astype("float32")
    ds["thetao"].attrs = {"long_name": "sea water potential temperature", "units": "degC"}
    ds["thetao"].encoding = {"dtype": "float32", "_FillValue": np.nan, "compressor": compressor}

    expected = pd.date_range(f"{year}-01-01", periods=ds.sizes["time"], freq="D")
    if not np.array_equal(ds["time"].values, expected.values):
        raise RuntimeError(f"{year}: the source time axis is not a gap-free daily series")
    ds = ds.chunk({"time": time_chunk, "depth": -1, "lat": -1, "lon": -1})

    markers = Path(f"{store}.blocks")
    markers.mkdir(exist_ok=True)
    if not (store / ".zmetadata").exists():
        ds.to_zarr(store, mode="w", compute=False, consolidated=True)  # coordinates and metadata only

    n_time = ds.sizes["time"]
    n_blocks = -(-n_time // block_days)
    for block in range(n_blocks):
        marker = markers / f"{block:04d}"
        if marker.exists():
            continue
        days = slice(block * block_days, min((block + 1) * block_days, n_time))
        ds.isel(time=days).drop_vars(["depth", "lat", "lon", "time"]).to_zarr(store, region={"time": days})
        marker.touch()
    done.touch()
    return "downloaded"

In [ ]:
for year in years:
    start = time.time()
    status = fetch_year(year)
    print(f"{year}: {status} ({time.time() - start:.0f} s)", flush=True)

## 3. The Ocean Domain

`valid` is the static sea mask and `thetao` finite on every downloaded day. Land and sea floor are False at each level.

In [ ]:
valid_store = input_dir / "glorys12_valid.zarr"
if not valid_store.exists():
    valid = static["mask"].astype(bool).compute()
    for year in years:
        finite = xr.open_zarr(input_dir / f"thetao_{year}.zarr")["thetao"].notnull().all("time").compute()
        valid = valid & finite.assign_coords(depth=valid.depth)
    valid.rename("valid").to_dataset().to_zarr(valid_store, mode="w", consolidated=True)

valid = xr.open_zarr(valid_store).valid
print(f"{int(valid.sum())} valid voxels, of {valid.size} in the box")